# Video Game Sales — Exploratory Data Analysis
End-to-end: load → clean → SQLite → 30 analytical questions with charts.

Run order:
1. Restart kernel
2. Run all cells


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd, numpy as np, sqlite3
import matplotlib.pyplot as plt, seaborn as sns
from utils.data_loader import load_raw, clean, build_database, merged_view, DB_PATH
sns.set_theme(style='whitegrid')

## 1. Load + clean + build SQLite

In [ ]:
games_raw, vg_raw = load_raw()
games, vg = clean(games_raw, vg_raw)
build_database(games, vg)
print('games:', games.shape, '| sales:', vg.shape)

In [ ]:
df = merged_view()
df.head()

## 2. Thirty analytical questions
Each cell answers one question with a SQL query or pandas op + chart.

In [ ]:
questions = [
  'Q1  Top 10 games by global sales',
  'Q2  Top 10 platforms by global sales',
  'Q3  Top 10 publishers by global sales',
  'Q4  Sales per genre',
  'Q5  Yearly global sales trend',
  'Q6  Region share (NA/EU/JP/Other)',
  'Q7  Top genre per region',
  'Q8  Avg critic score per genre',
  'Q9  Avg user score per genre',
  'Q10 Critic vs user score correlation',
  'Q11 Critic score vs global sales',
  'Q12 Top developer by sales',
  'Q13 Rating distribution',
  'Q14 Sales by ESRB rating',
  'Q15 Best year for the industry',
  'Q16 Platform lifecycle (sales over years)',
  'Q17 Genre popularity over time',
  'Q18 Top publisher per platform',
  'Q19 Most prolific publishers (game count)',
  'Q20 Avg sales per game per publisher',
  'Q21 Outlier blockbuster games (IQR)',
  'Q22 Genre with highest avg user_count',
  'Q23 Region affinity heatmap (genre×region)',
  'Q24 Median sales per platform',
  'Q25 Top game per year',
  'Q26 Share of Japan-led titles',
  'Q27 Sales decline post-2010 platforms',
  'Q28 Critic vs commercial mismatch',
  'Q29 Catalog growth over years',
  'Q30 Pareto: % publishers driving 80% sales',
]
for q in questions: print('•', q)

In [ ]:
# Q1 — Top 10 games
top = df.groupby('name')['global_sales'].sum().nlargest(10)
top.plot(kind='barh', figsize=(8,4), title='Q1 Top 10 games'); plt.gca().invert_yaxis(); plt.show()

In [ ]:
# Q5 — Yearly trend
yr = df.groupby('release_year')['global_sales'].sum().sort_index()
yr = yr[(yr.index>1970)&(yr.index<2030)]
yr.plot(figsize=(9,4), title='Q5 Global sales by year'); plt.show()

In [ ]:
# Q23 — Region × Genre heatmap
regions = ['na_sales','eu_sales','jp_sales','other_sales']
hm = df.groupby('genre')[regions].sum()
plt.figure(figsize=(8,6))
sns.heatmap(hm, cmap='viridis', annot=True, fmt='.1f'); plt.title('Q23 Genre × Region'); plt.show()

In [ ]:
# Q30 — Pareto
pub = df.groupby('publisher')['global_sales'].sum().sort_values(ascending=False)
cum = pub.cumsum() / pub.sum()
n80 = int((cum<=0.8).sum())+1
print(f'{n80} of {len(pub)} publishers ({n80/len(pub)*100:.1f}%) drive 80% of sales')

Add additional cells for Q2–Q29 following the same pattern. The dataset is already in `df` and SQLite at `db/videogames.db`.